# Ejercicio 3 - Consultas directas sobre archivos Parquet

Este notebook ejecuta las consultas de `sql/ex3_exploracion.sql` directamente sobre los
archivos Parquet de `data/raw/` (sin importarlos a una tabla). El archivo SQL es la fuente
unica de las consultas; la documentacion de cada una esta en `docs/ejercicio3.md`.

Requisito: haber descargado los datos con `python scripts/download_data.py`.

In [1]:
import os, sys, time
from pathlib import Path
import duckdb
import pandas as pd

# Trabajar desde la raiz del proyecto: las rutas del SQL son relativas a ella.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "scripts"))
from run_sql import dividir_bloques

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)

preparacion, bloques = dividir_bloques(Path("sql/ex3_exploracion.sql").read_text(encoding="utf-8"))
con = duckdb.connect()          # base en memoria: nada se materializa
con.execute(preparacion)         # vistas yellow / green sobre read_parquet()
consultas = dict(bloques)
print(f"DuckDB {duckdb.__version__} - {len(consultas)} consultas cargadas")
for titulo in consultas:
    print(" ", titulo)

DuckDB 1.5.5 - 18 consultas cargadas
  Q3.1 Cantidad de archivos disponibles por tipo y anio
  Q3.2a Cantidad de registros segun los metadatos Parquet (sin leer filas)
  Q3.2b Cantidad de registros por archivo (conteo leyendo los datos)
  Q3.3 Columnas presentes y en cuantos archivos aparece cada una
  Q3.4a Tipos de datos de las columnas (taxis amarillos)
  Q3.4b Tipos de datos de las columnas (taxis verdes)
  Q3.4c Archivos que contienen la columna request_source
  Q3.5a Muestra de registros (amarillos)
  Q3.5b Muestra de registros (verdes)
  Q3.6a Valores nulos por columna (amarillos)
  Q3.6b Valores nulos por columna (verdes)
  Q3.6c Registros con fecha de abordaje fuera del mes del archivo
  Q3.6d Inconsistencias de tiempo, distancia, pasajeros y montos
  Q3.6e Distribucion de valores extremos de distancia y monto total
  Q3.6f Codigos categoricos fuera del diccionario de la TLC
  Q3.6g Registros completamente duplicados
  Q3.6h Identificadores de zona fuera de las zonas validas (

In [2]:
def ejecutar(prefijo: str) -> pd.DataFrame:
    """Ejecuta la consulta cuyo titulo empieza con `prefijo` y muestra su SQL y tiempo."""
    titulo, sql = next((t, s) for t, s in consultas.items() if t.startswith(prefijo + " "))
    inicio = time.perf_counter()
    df = con.sql(sql).df()
    print(f"{titulo}  ({time.perf_counter() - inicio:.2f} s)")
    return df

## 3.1 Cantidad de archivos

In [3]:
ejecutar("Q3.1")

Q3.1 Cantidad de archivos disponibles por tipo y anio  (0.07 s)


,tipo,anio,archivos,mib
0,green,2026,8,7.9
1,yellow,2026,8,487.8


## 3.2 Cantidad de registros
Primero solo con los metadatos del footer Parquet (no lee filas) y luego contando las filas por archivo.

In [4]:
ejecutar("Q3.2a")

Q3.2a Cantidad de registros segun los metadatos Parquet (sin leer filas)  (0.04 s)


,tipo,registros
0,green,337114.0
1,yellow,29703355.0


In [5]:
ejecutar("Q3.2b")

Q3.2b Cantidad de registros por archivo (conteo leyendo los datos)  (0.10 s)


,tipo,mes,registros
0,yellow,2026-01,3724889
1,yellow,2026-02,3399866
2,yellow,2026-03,3952451
3,yellow,2026-04,3831240
4,yellow,2026-05,4090836
5,yellow,2026-06,3837248
6,yellow,2026-07,3530109
7,yellow,2026-08,3336716
8,green,2026-01,40272
9,green,2026-02,37373


## 3.3 Columnas presentes

In [6]:
ejecutar("Q3.3")

Q3.3 Columnas presentes y en cuantos archivos aparece cada una  (0.04 s)


,tipo,columna,archivos_con_columna
0,yellow,request_source,3
1,yellow,Airport_fee,8
2,yellow,DOLocationID,8
3,yellow,PULocationID,8
4,yellow,RatecodeID,8
5,yellow,VendorID,8
6,yellow,cbd_congestion_fee,8
7,yellow,congestion_surcharge,8
8,yellow,extra,8
9,yellow,fare_amount,8


## 3.4 Tipos de datos

In [7]:
ejecutar("Q3.4a")

Q3.4a Tipos de datos de las columnas (taxis amarillos)  (0.03 s)


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [8]:
ejecutar("Q3.4b")

Q3.4b Tipos de datos de las columnas (taxis verdes)  (0.03 s)


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [9]:
ejecutar("Q3.4c")

Q3.4c Archivos que contienen la columna request_source  (0.04 s)


,archivo,tiene_request_source,columnas
0,green_tripdata_2026-01,0,21
1,green_tripdata_2026-02,0,21
2,green_tripdata_2026-03,0,21
3,green_tripdata_2026-04,0,21
4,green_tripdata_2026-05,0,21
5,green_tripdata_2026-06,1,22
6,green_tripdata_2026-07,1,22
7,green_tripdata_2026-08,1,22
8,yellow_tripdata_2026-01,0,20
9,yellow_tripdata_2026-02,0,20


## 3.5 Muestra de registros

In [10]:
ejecutar("Q3.5a")

Q3.5a Muestra de registros (amarillos)  (0.23 s)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-01-01 00:34:45,2026-01-01 00:40:15,4,0.90,1,N,262,141,1,7.9,3.50,0.5,3.20,0.00,1.0,16.10,2.5,0.0,0.00,None
1,1,2026-01-01 00:06:07,2026-01-01 00:20:20,2,1.90,1,N,231,90,1,14.2,4.25,0.5,3.95,0.00,1.0,23.90,2.5,0.0,0.75,None
2,2,2026-01-01 03:15:42,2026-01-01 03:20:33,1,1.15,1,N,263,140,1,7.2,1.00,0.5,2.44,0.00,1.0,14.64,2.5,0.0,0.00,None
3,2,2026-01-01 04:25:35,2026-01-01 04:38:18,1,3.69,1,N,107,143,1,17.7,1.00,0.5,4.69,0.00,1.0,28.14,2.5,0.0,0.75,None
4,2,2026-01-01 06:33:56,2026-01-01 06:55:37,2,10.27,2,N,68,138,2,70.0,0.00,0.5,0.00,6.94,1.0,81.69,2.5,0.0,0.75,None


In [11]:
ejecutar("Q3.5b")

Q3.5b Muestra de registros (verdes)  (0.10 s)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-02 05:43:34,2026-01-02 05:44:09,N,5,34,34,2,0.40,70.0,0.0,0.0,14.20,0.0,NaN,1.0,85.20,1,2,0.0,0.0,None
1,2,2026-01-02 21:02:00,2026-01-02 21:07:42,N,1,74,41,1,1.17,7.9,1.0,0.5,2.08,0.0,NaN,1.0,12.48,1,1,0.0,0.0,None
2,1,2026-01-14 21:03:03,2026-01-14 21:06:16,N,1,181,65,1,0.60,5.8,1.0,1.5,0.00,0.0,NaN,1.0,8.30,2,1,0.0,0.0,None
3,2,2026-01-16 19:11:20,2026-01-16 19:29:44,N,1,74,151,1,2.33,17.7,2.5,0.5,4.34,0.0,NaN,1.0,26.04,1,1,0.0,0.0,None
4,2,2026-01-19 02:15:45,2026-01-19 02:20:57,N,1,95,102,1,1.35,8.6,1.0,0.5,1.00,0.0,NaN,1.0,12.10,1,1,0.0,0.0,None


## 3.6 Problemas de calidad de datos

In [12]:
ejecutar("Q3.6a")

Q3.6a Valores nulos por columna (amarillos)  (0.42 s)


,registros,null_passenger_count,null_ratecodeid,null_store_and_fwd_flag,null_congestion_surcharge,null_airport_fee,null_request_source,pct_null_passenger_count
0,29703355,7716688,7716688,7716688,7716688,7716688,26799909,25.98


In [13]:
ejecutar("Q3.6b")

Q3.6b Valores nulos por columna (verdes)  (0.06 s)


,registros,null_passenger_count,null_ratecodeid,null_payment_type,null_trip_type,null_congestion_surcharge,null_ehail_fee,null_request_source
0,337114,48775,48775,48775,48777,48775,337114,317903


In [14]:
ejecutar("Q3.6i")

Q3.6i Origen de los registros con nulos (mismas filas en varias columnas)  (0.51 s)


,tipo,VendorID,payment_type,registros_con_nulos,nulos_simultaneos,total_promedio
0,yellow,2,0,6761917,6761917,32.88
1,yellow,1,0,895381,895381,30.28
2,yellow,6,0,59390,59390,31.74
3,green,6,<NA>,34847,34847,29.41
4,green,2,<NA>,13400,13400,34.20
5,green,1,<NA>,528,528,24.30


In [15]:
ejecutar("Q3.6c")

Q3.6c Registros con fecha de abordaje fuera del mes del archivo  (0.89 s)


,tipo,fuera_de_mes,pickup_min,pickup_max
0,yellow,146,2001-01-01 09:23:58,2026-08-31 23:59:59
1,green,98,2008-12-31 17:35:31,2026-08-31 23:58:28


In [16]:
ejecutar("Q3.6d").T

Q3.6d Inconsistencias de tiempo, distancia, pasajeros y montos  (1.96 s)


,0,1
tipo,yellow,green
registros,29703355,337114
dropoff_antes_pickup,10,5
duracion_cero,371673,229
duracion_mayor_24h,263,4
distancia_cero,952231,12212
distancia_negativa,0,0
distancia_mayor_100mi,1223,72
pasajeros_cero,91359,4527
pasajeros_mas_6,28,99


In [17]:
ejecutar("Q3.6e")

Q3.6e Distribucion de valores extremos de distancia y monto total  (3.94 s)


,tipo,dist_min,dist_p50_p99_p999,dist_max,total_min,total_p50_p99_p999,total_max
0,yellow,0.0,"[1.86, 19.5, 30.736460000015796]",328522.20,-2560.2,"[23.58, 105.75, 184.42]",7053.5
1,green,0.0,"[2.07, 17.76, 30.05]",179830.92,-501.5,"[20.46, 97.2, 224.77513999999735]",1678.2


In [18]:
ejecutar("Q3.6f")

Q3.6f Codigos categoricos fuera del diccionario de la TLC  (0.96 s)


,tipo,columna,valor,registros
0,yellow,RatecodeID,NaN,7716688
1,yellow,RatecodeID,1,20102072
2,yellow,RatecodeID,2,694936
3,yellow,RatecodeID,3,90052
4,yellow,RatecodeID,4,67285
5,yellow,RatecodeID,5,262614
6,yellow,RatecodeID,6,15
7,yellow,RatecodeID,99,769693
8,yellow,VendorID,1,5467071
9,yellow,VendorID,2,23809774


In [19]:
ejecutar("Q3.6g")

Q3.6g Registros completamente duplicados  (7.69 s)


,tipo,duplicados
0,yellow,7
1,green,0


In [20]:
ejecutar("Q3.6h")

Q3.6h Identificadores de zona fuera de las zonas validas (1-263; 264-265 = desconocido)  (0.45 s)


,tipo,pu_fuera_de_zonas,do_fuera_de_zonas,pu_desconocido,do_desconocido
0,yellow,49676,178136,49676,178136
1,green,1131,5600,1131,5600


## 3.9 Plan de ejecucion: lectura directa del Parquet

`EXPLAIN ANALYZE` muestra que DuckDB lee el Parquet con `READ_PARQUET` y solo las columnas
que la consulta necesita (*projection pushdown*); el filtro se evalua durante el escaneo.

In [21]:
plan = con.sql("""
    EXPLAIN ANALYZE
    SELECT count(*) FROM read_parquet('data/raw/yellow/*/*.parquet')
    WHERE trip_distance > 100
""").fetchall()
print(plan[0][1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE     SELECT count(*) FROM read_parquet('data/raw/yellow/*/*.parquet')     WHERE trip_distance > 100 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.191s              ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│        Aggregates:        │
│   